# MCP SQL Query Expert (GCP-native) — v3 (shows SQL + validation)

Same as v2, plus: it **prints the exact SQL** each tool runs, and **optionally validates that SQL** with the SQL Expert's guardrails so you can see what's checked.

### What validation actually happens in the MCP path

| Layer | What it does |
|---|---|
| **Toolbox parameterized queries** | Parameter values are *bound*, never string-concatenated → **SQL injection is prevented**. |
| **`curated` mode** | The SQL is **fixed** (you wrote it). The model only picks a tool + parameters. Nothing to inject, nothing to validate. |
| **`prebuilt` mode** | `execute_sql` runs **whatever SQL the model writes**, limited only by the DB user's permissions. |
| **`sql_expert/validators.py`** | **NOT used by MCP.** No read-only check, table/column allowlist, `EXPLAIN`, or row cap — those live only in the `sql_expert` pipeline. |

This notebook adds an **optional** validation step (Step 5b) that runs each query through `validate_sql()` so you can *see* whether it would pass those guardrails. In `prebuilt` mode that's genuinely useful (it flags unsafe model-written SQL); in `curated` mode it just confirms your fixed queries are fine.

> Note: the discovery tools query `information_schema`, which the allowlist flags as "not a business table." That's expected and harmless — those are safe system catalogs, not your data tables.

### How this notebook is built

Self-contained by design — one `.ipynb` you can hand to someone. That costs some verbosity in the cells, and it means the code here has to be defensive about things a library would normally handle:

* **Secrets never reach disk.** `tools.yaml` stores `${TOOLBOX_PG_PASSWORD}`; Toolbox expands it from its own environment. The file is written to a temp directory and deleted in Step 7. (`SECRET_MODE = "inline"` in Step 3 is the fallback if your Toolbox build doesn't expand `${...}`.)
* **The server's stdout goes to a log file, never to a `PIPE`.** An undrained pipe fills at ~64 KB and the child then blocks forever on its next log line — a hang with no error message.
* **The port is OS-assigned.** A hardcoded port that's already in use makes the readiness check pass against *someone else's* server while yours dies silently.
* **The binary download is version-stamped and atomic**, so bumping the version actually re-downloads and a failed download can't leave a truncated executable to be run.
* **Anything interpolated into SQL is validated first** (`safe_identifier`) — Postgres can't bind a schema name as a parameter, so validation is the only defence there.
* **Cleanup is idempotent and registered with `atexit`**, so re-running a cell or crashing mid-notebook doesn't leak processes.

Every cell is safe to re-run.

## Step 0 — Install dependencies

In [ ]:
# Client libraries. `pyyaml` writes tools.yaml; `asyncpg` is used only in Step 5b.
%pip install -q toolbox-langchain langchain-google-vertexai langgraph asyncpg python-dotenv pyyaml

# Record what this run actually resolved to. Unpinned installs mean a library release
# can break the notebook without a line of it changing — freeze these numbers into
# requirements.txt once you have a combination that works.
from importlib.metadata import PackageNotFoundError, version

for _pkg in ("toolbox-langchain", "langchain-google-vertexai", "langgraph",
             "langchain-core", "asyncpg", "pyyaml"):
    try:
        print(f"{_pkg:<28} {version(_pkg)}")
    except PackageNotFoundError:
        print(f"{_pkg:<28} NOT INSTALLED")

## Step 1 — Config (and the MODE switch)

In [ ]:
import pathlib
import re
import sys


def find_backend_root(start=None, marker="config.py", max_levels=6):
    """Walk up from `start` looking for the directory that holds `marker`.

    Also checks <dir>/backend_py and <dir>/backend_py/backend_py at each level, so
    this works whether the notebook was launched from the repo root, from
    backend_py, or from a sibling folder like new_python_codes/mcp_server.
    """
    origin = pathlib.Path(start or pathlib.Path.cwd()).resolve()
    for level, directory in enumerate([origin, *origin.parents]):
        if level > max_levels:
            break
        for candidate in (directory, directory / "backend_py",
                          directory / "backend_py" / "backend_py"):
            if (candidate / marker).is_file():
                return candidate.resolve()
    raise FileNotFoundError(
        f"{marker} not found within {max_levels} levels of {origin}. Set it by hand:\n"
        f"    BACKEND = pathlib.Path(r'C:/Sysco/backend_py/backend_py')"
    )


BACKEND = find_backend_root()
if str(BACKEND) not in sys.path:
    sys.path.insert(0, str(BACKEND))

from config import get_settings

settings = get_settings()

# ===================== CHOOSE YOUR MODE HERE =====================
#   "curated"  = you author fixed parameterized queries in tools.yaml (safe).
#   "prebuilt" = Toolbox auto-generates generic tools incl. free-form execute_sql (unguarded).
MODE = "curated"
# ================================================================
assert MODE in ("curated", "prebuilt"), MODE

_IDENTIFIER = re.compile(r"^[A-Za-z_][A-Za-z0-9_]{0,62}$")


def safe_identifier(value, kind="identifier"):
    """Return `value` if it is a plain SQL identifier, else raise.

    Postgres cannot bind a schema or table name as a parameter, so anything
    interpolated into a statement has to be validated instead of escaped.
    """
    if not isinstance(value, str) or not _IDENTIFIER.match(value):
        raise ValueError(f"unsafe {kind}: {value!r} (letters, digits, underscore only)")
    return value


def setting(name, default=None):
    """Read one settings field, failing with the env var to set rather than AttributeError."""
    value = getattr(settings, name, None)
    if value not in (None, ""):
        return value
    if default is not None:
        return default
    raise ValueError(
        f"Setting {name!r} is missing or empty. Add it to {BACKEND / 'config.py'} "
        f"and set {name.upper()} in your .env / Secret Manager."
    )


def mask(value, keep=3):
    """Notebook outputs get committed — treat everything printed here as public."""
    text = str(value or "")
    return text[:keep] + "*" * max(0, len(text) - keep)


DB = {
    "host": setting("sql_expert_db_host"),
    "port": int(setting("sql_expert_db_port", 5432)),
    "database": setting("sql_expert_db_name"),
    "user": setting("sql_expert_db_user"),
    "password": setting("sql_expert_db_password").get_secret_value(),
}
SCHEMA = safe_identifier(setting("sql_expert_db_schema", "public"), "schema name")
GCP_PROJECT = setting("gcp_project")
GCP_LOCATION = setting("gcp_location", "us-central1")
GEMINI_MODEL = setting("vertex_model", "gemini-2.5-flash")

print(f"MODE   : {MODE}")
print(f"DB     : {mask(DB['user'])}@{mask(DB['host'])}:{DB['port']}"
      f"/{DB['database']}  (schema={SCHEMA})")
print(f"Vertex : project={mask(GCP_PROJECT, 4)} location={GCP_LOCATION} model={GEMINI_MODEL}")

## Step 2 — Download the MCP Toolbox server

In [ ]:
import hashlib
import os
import platform
import stat
import urllib.request

TOOLBOX_VERSION = "1.7.0"   # pin; check the repo Releases for the latest
TOOLBOX_SHA256 = None       # set once you've recorded the digest for your platform


def toolbox_asset(system=None, machine=None):
    """Return (release_subdirectory, filename) for this platform."""
    system = (system or platform.system()).lower()
    machine = (machine or platform.machine()).lower()
    arch = "arm64" if machine in ("arm64", "aarch64") else "amd64"
    if system == "windows":
        return "windows/amd64", "toolbox.exe"
    if system == "darwin":
        return f"darwin/{arch}", "toolbox"
    return f"linux/{arch}", "toolbox"


def sha256_of(path, chunk=1 << 20):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(chunk), b""):
            digest.update(block)
    return digest.hexdigest()


def download_toolbox(dest_dir, version=TOOLBOX_VERSION, sha256=TOOLBOX_SHA256, force=False):
    """Fetch the Toolbox binary.

    Three details that matter:
      * the filename carries the version, so bumping `version` re-downloads instead
        of silently reusing the old binary;
      * the download lands on a `.part` file and is renamed only on success, so a
        failed transfer can never leave a truncated executable behind;
      * the checksum is verified when you supply one — this is a file you are about
        to execute.
    """
    release_dir, filename = toolbox_asset()
    dest_dir = pathlib.Path(dest_dir)
    dest_dir.mkdir(parents=True, exist_ok=True)
    target = (dest_dir / f"toolbox-{version}{pathlib.Path(filename).suffix}").resolve()

    if target.exists() and not force:
        if sha256 and sha256_of(target) != sha256:
            raise RuntimeError(f"checksum mismatch for cached binary {target}")
        return target

    url = (f"https://storage.googleapis.com/mcp-toolbox-for-databases"
           f"/v{version}/{release_dir}/{filename}")
    partial = target.with_name(target.name + ".part")
    print("downloading", url)
    try:
        urllib.request.urlretrieve(url, partial)
        if sha256:
            actual = sha256_of(partial)
            if actual != sha256:
                raise RuntimeError(f"checksum mismatch: expected {sha256}, got {actual}")
        if os.name != "nt":
            partial.chmod(partial.stat().st_mode | stat.S_IEXEC)
        partial.replace(target)   # atomic
    finally:
        partial.unlink(missing_ok=True)
    return target


# Kept out of the repo root; add `.toolbox/` to .gitignore if you move it.
TOOLBOX_BIN = download_toolbox(BACKEND / ".toolbox")
print("binary:", TOOLBOX_BIN)

## Step 3 — Author `tools.yaml` (curated mode)

Each tool is a **fixed, parameterized SQL statement**. The model chooses *which* tool and *what* parameters — never the SQL. We keep the `TOOL_CONFIG` dict around so Step 6 can show you the exact SQL behind each tool call.

Two things worth noticing:

* the file contains `password: ${TOOLBOX_PG_PASSWORD}`, not your password — Toolbox expands it from its own environment, so the secret only ever exists in process memory;
* the file is written to a **temp directory** (removed in Step 7), so it can't be committed by accident. If you'd rather keep it in the repo, add `tools.yaml` to `.gitignore` first.

The cell ends with an assertion that every `$n` placeholder has a declared parameter — an authoring slip you'd otherwise discover when the model calls the tool.

In [ ]:
import tempfile

import yaml

SOURCE_NAME = "rfp-analytics"
CURATED_TOOLSET = "rfp-aiq"
PASSWORD_ENV_VAR = "TOOLBOX_PG_PASSWORD"

# "env"    -> tools.yaml stores ${TOOLBOX_PG_PASSWORD} and Toolbox expands it from
#             its environment: no secret on disk. Recommended.
# "inline" -> literal password in the file. Fallback for Toolbox builds that don't
#             do ${...} expansion (symptom: auth fails using the literal string).
SECRET_MODE = "env"

# Everything this run writes lives here and is deleted in Step 7.
RUN_DIR = pathlib.Path(tempfile.mkdtemp(prefix="toolbox-"))


def build_tool_config(schema, db):
    """3 parts: sources (DB connections), tools (fixed SQL), toolsets (named groups)."""
    schema = safe_identifier(schema, "schema name")
    password = ("${" + PASSWORD_ENV_VAR + "}") if SECRET_MODE == "env" else db["password"]
    return {
        "sources": {
            SOURCE_NAME: {
                "kind": "postgres",
                "host": db["host"], "port": db["port"], "database": db["database"],
                "user": db["user"], "password": password,
            }
        },
        "tools": {
            "list_tables": {
                "kind": "postgres-sql", "source": SOURCE_NAME,
                "description": ("List all base tables in the analytics database. "
                                "Call this first to discover what data exists."),
                "statement": (
                    "SELECT table_name FROM information_schema.tables "
                    f"WHERE table_schema = '{schema}' AND table_type = 'BASE TABLE' "
                    "ORDER BY table_name;"
                ),
            },
            "describe_table": {
                "kind": "postgres-sql", "source": SOURCE_NAME,
                "description": "Return the columns and data types of one table.",
                "parameters": [{"name": "table_name", "type": "string",
                                "description": "Exact table name, e.g. 'bids'."}],
                "statement": (
                    "SELECT column_name, data_type FROM information_schema.columns "
                    f"WHERE table_schema = '{schema}' AND table_name = $1 "
                    "ORDER BY ordinal_position;"
                ),
            },
            "total_bids": {
                "kind": "postgres-sql", "source": SOURCE_NAME,
                "description": ("Total number of bids across current and historical "
                                "tables (bids + bids_old)."),
                "statement": (
                    "SELECT count(*) AS total_bids FROM "
                    "(SELECT 1 FROM bids UNION ALL SELECT 1 FROM bids_old) t;"
                ),
            },
            "bids_by_status": {
                "kind": "postgres-sql", "source": SOURCE_NAME,
                "description": ("Count bids with a given status, combining current "
                                "and historical tables."),
                "parameters": [{"name": "status", "type": "string",
                                "description": "Bid status to filter on, e.g. 'Complete'."}],
                "statement": (
                    "SELECT count(*) AS n FROM "
                    "(SELECT status FROM bids UNION ALL SELECT status FROM bids_old) t "
                    "WHERE t.status = $1;"
                ),
            },
            "total_suppliers": {
                "kind": "postgres-sql", "source": SOURCE_NAME,
                "description": ("Total number of suppliers across current and "
                                "historical tables (suppliers + suppliers_old)."),
                "statement": (
                    "SELECT count(*) AS total_suppliers FROM "
                    "(SELECT 1 FROM suppliers UNION ALL SELECT 1 FROM suppliers_old) t;"
                ),
            },
        },
        "toolsets": {
            CURATED_TOOLSET: ["list_tables", "describe_table", "total_bids",
                              "bids_by_status", "total_suppliers"],
        },
    }


TOOL_CONFIG = build_tool_config(SCHEMA, DB)

# Self-check: every $n placeholder must have a declared parameter, and vice versa.
for _name, _spec in TOOL_CONFIG["tools"].items():
    _used = {int(n) for n in re.findall(r"\$(\d+)", _spec["statement"])}
    assert _used == set(range(1, len(_spec.get("parameters", [])) + 1)), \
        f"{_name}: placeholders {_used} do not match declared parameters"
assert set(TOOL_CONFIG["toolsets"][CURATED_TOOLSET]) <= set(TOOL_CONFIG["tools"])

TOOLS_YAML = RUN_DIR / "tools.yaml"
TOOLS_YAML.write_text(yaml.safe_dump(TOOL_CONFIG, sort_keys=False), encoding="utf-8")
if os.name != "nt":
    TOOLS_YAML.chmod(0o600)

# Show the source block so you can see how the password is stored — redacted if you
# switched to inline mode, because this output gets saved into the .ipynb.
_shown = dict(TOOL_CONFIG["sources"][SOURCE_NAME])
if SECRET_MODE != "env":
    _shown["password"] = "<redacted>"
_shown["host"] = mask(_shown["host"])
print(f"Wrote {TOOLS_YAML}  (secret mode: {SECRET_MODE}, {len(TOOL_CONFIG['tools'])} tools)")
print(yaml.safe_dump({SOURCE_NAME: _shown}, sort_keys=False))

## Step 4 — Launch the Toolbox MCP server

In [ ]:
import atexit
import socket
import subprocess
import time

proc = globals().get("proc")               # may survive a re-run of this cell
log_handle = globals().get("log_handle")


def port_in_use(port, host="127.0.0.1", timeout=0.5):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(timeout)
        return sock.connect_ex((host, port)) == 0


def pick_free_port(host="127.0.0.1"):
    """Let the OS choose. A hardcoded port that is already taken makes the readiness
    check pass against someone else's server while ours dies with 'address in use'."""
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.bind((host, 0))
        return sock.getsockname()[1]


def stop_toolbox():
    """Idempotent — safe to call twice, from Step 7, or from atexit."""
    global proc, log_handle
    if isinstance(proc, subprocess.Popen) and proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=5)
        except subprocess.TimeoutExpired:
            proc.kill()
            proc.wait(timeout=5)
    proc = None
    if log_handle is not None and not log_handle.closed:
        log_handle.close()
    log_handle = None


def log_tail(lines=40):
    if not LOG_PATH.exists():
        return "(no log)"
    text = LOG_PATH.read_text(encoding="utf-8", errors="replace")
    return "\n".join(text.splitlines()[-lines:]) or "(log empty)"


atexit.register(stop_toolbox)   # a forgotten kernel must not leave the server running
stop_toolbox()                  # clean restart if you re-run this cell

TOOLBOX_PORT = pick_free_port()
TOOLBOX_URL = f"http://127.0.0.1:{TOOLBOX_PORT}"
LOG_PATH = RUN_DIR / f"toolbox-{TOOLBOX_PORT}.log"

args = [str(TOOLBOX_BIN), "--address", "127.0.0.1", "--port", str(TOOLBOX_PORT)]
env = os.environ.copy()

if MODE == "curated":
    args += ["--config", str(TOOLS_YAML)]     # some builds spell this --tools-file
    TOOLSET_NAME = CURATED_TOOLSET
    env[PASSWORD_ENV_VAR] = DB["password"]    # what ${TOOLBOX_PG_PASSWORD} expands to
else:
    args += ["--prebuilt", "postgres"]        # auto-generate generic tools
    TOOLSET_NAME = None
    env.update({
        "POSTGRES_HOST": str(DB["host"]), "POSTGRES_PORT": str(DB["port"]),
        "POSTGRES_DATABASE": str(DB["database"]), "POSTGRES_USER": str(DB["user"]),
        "POSTGRES_PASSWORD": DB["password"],
    })

# stdout goes to a FILE, never to subprocess.PIPE. Nothing in this notebook drains a
# pipe, and once the OS buffer fills (~64 KB) the server blocks forever on its next
# log line — a hang with no error anywhere.
log_handle = LOG_PATH.open("w+", encoding="utf-8", errors="replace")
proc = subprocess.Popen(args, env=env, stdout=log_handle, stderr=subprocess.STDOUT)

STARTUP_TIMEOUT = 30
deadline = time.monotonic() + STARTUP_TIMEOUT
while True:
    if proc.poll() is not None:
        raise RuntimeError(f"Toolbox exited during startup (code {proc.returncode}):\n"
                           f"{log_tail()}")
    if port_in_use(TOOLBOX_PORT):
        break
    if time.monotonic() > deadline:
        stop_toolbox()
        raise RuntimeError(f"Toolbox never listened on :{TOOLBOX_PORT} within "
                           f"{STARTUP_TIMEOUT}s:\n{log_tail()}")
    time.sleep(0.25)

print(f"OK  Toolbox MCP server ({MODE}) at {TOOLBOX_URL}")
print(f"    log: {LOG_PATH}")

## Step 5 — Load the tools into a Gemini (Vertex) agent

In [ ]:
import inspect

from langchain_core.messages import HumanMessage, SystemMessage
from langchain_google_vertexai import ChatVertexAI
from langgraph.prebuilt import create_react_agent
from toolbox_langchain import ToolboxClient

tb = ToolboxClient(TOOLBOX_URL)


async def load_toolset(name):
    """toolbox-langchain renamed load_toolset -> aload_toolset; accept either."""
    loader = getattr(tb, "aload_toolset", None) or getattr(tb, "load_toolset", None)
    if loader is None:
        raise RuntimeError("ToolboxClient exposes no (a)load_toolset method")
    result = loader() if name is None else loader(name)
    return await result if inspect.isawaitable(result) else result


tools = await load_toolset(TOOLSET_NAME)
if not tools:
    raise RuntimeError(f"Toolbox returned no tools. Check tools.yaml and:\n{log_tail(20)}")
print("Loaded MCP tools:", [t.name for t in tools])

SYSTEM = (
    "You are RFP AIQ, the Sysco Bid Intelligence assistant. Answer the user's question "
    "by calling the provided database tools. Use the discovery tools first if you need "
    "the schema. Reply with ONE short, human sentence. Never show SQL, tool names, JSON, "
    "or row dumps."
)

llm = ChatVertexAI(model=GEMINI_MODEL, project=GCP_PROJECT, location=GCP_LOCATION,
                   temperature=0, max_retries=3)   # retries cover Vertex 429/503

# The system prompt belongs to the agent, not to every call site. Older langgraph
# releases have no `prompt` kwarg, so fall back to prepending it per call.
try:
    agent = create_react_agent(llm, tools, prompt=SYSTEM)
    PROMPT_IN_AGENT = True
except TypeError:
    agent = create_react_agent(llm, tools)
    PROMPT_IN_AGENT = False

print(f"OK  Gemini agent ready (system prompt held by agent: {PROMPT_IN_AGENT})")

## Step 5b — Show the SQL & (optionally) validate it

This builds a small schema catalog (table → columns) via `asyncpg`, then wraps the SQL Expert's `validate_sql()`. It also defines `show_sql()` which reconstructs the exact query behind each tool call. **The MCP Toolbox does not run this validator — we call it here only so you can see the verdict.**

`validate_sql()` checks: read-only (SELECT/WITH only, no INSERT/UPDATE/DROP…), single statement, every table exists, every `alias.column` exists. It returns `(ok, reason)`.

In [ ]:
# The guardrail lives in the sql_expert pipeline. Import it optionally so the notebook
# still runs in a checkout that doesn't ship it.
try:
    from services.sql_expert.validators import validate_sql
except ImportError as exc:
    validate_sql = None
    print(f"validator unavailable ({exc}) — verdicts will show n/a")

# --- Build the table/column catalog the validator needs (one query, not one per table) ---
KNOWN_TABLES, TABLE_COLUMNS = set(), {}
try:
    import asyncpg

    _conn = await asyncpg.connect(host=DB["host"], port=DB["port"], database=DB["database"],
                                  user=DB["user"], password=DB["password"])
    try:
        _rows = await _conn.fetch(
            "SELECT c.table_name, c.column_name "
            "FROM information_schema.columns AS c "
            "JOIN information_schema.tables AS t "
            "  ON t.table_schema = c.table_schema AND t.table_name = c.table_name "
            "WHERE c.table_schema = $1 AND t.table_type = 'BASE TABLE'",
            SCHEMA,
        )
    finally:
        await _conn.close()

    for _row in _rows:
        TABLE_COLUMNS.setdefault(_row["table_name"].lower(), set()).add(
            _row["column_name"].lower())
    KNOWN_TABLES = set(TABLE_COLUMNS)
    print(f"catalog: {len(KNOWN_TABLES)} tables — validation ENABLED")
except Exception as exc:            # demo-only fallback: never block the notebook
    print(f"catalog unavailable ({type(exc).__name__}: {exc}) — verdicts will show n/a")


def validate(sql):
    """Return (ok, reason). ok is True/False, or None when we cannot judge."""
    if validate_sql is None or not KNOWN_TABLES:
        return None, "validator or catalog unavailable"
    return validate_sql(sql, KNOWN_TABLES, TABLE_COLUMNS)


def sql_literal(value):
    """Render a value as a SQL literal FOR DISPLAY ONLY.

    Never build a real query with this — the tools bind their parameters. repr() is
    not a SQL literal: it mangles quotes, None and booleans.
    """
    if value is None:
        return "NULL"
    if isinstance(value, bool):
        return "TRUE" if value else "FALSE"
    if isinstance(value, (int, float)):
        return repr(value)
    return "'" + str(value).replace("'", "''") + "'"


def show_sql(name, args):
    """Reconstruct the SQL behind a tool call so we can print it."""
    spec = TOOL_CONFIG["tools"].get(name) if MODE == "curated" else None
    if spec:
        statement = spec["statement"]
        # Highest index first, so $1 is not substituted inside $10.
        for index, param in reversed(list(enumerate(spec.get("parameters", []), start=1))):
            statement = statement.replace(f"${index}", sql_literal(args.get(param["name"])))
        return statement
    # prebuilt / generated: the SQL is one of the tool arguments.
    for key in ("sql", "query", "statement"):
        if isinstance(args.get(key), str):
            return args[key]
    return next((v for v in args.values()
                 if isinstance(v, str) and "select" in v.lower()), None)


print("show_sql() + validate() ready")

## Step 6 — Ask questions (now printing the SQL + validator verdict)

For every tool Gemini calls, `ask()` prints: the tool + args, the **actual SQL**, and the **validator verdict** (PASS / FAIL / n/a). Then the final one-line answer.

In [ ]:
MAX_AGENT_STEPS = 12   # caps the tool-calling loop so a confused model can't spin


def message_text(content):
    """Flatten a message's content to text — Gemini returns content blocks, not always str."""
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        return "".join(part if isinstance(part, str) else str(part.get("text", ""))
                       for part in content if isinstance(part, (str, dict)))
    return str(content)


async def ask(question: str) -> str:
    messages = [] if PROMPT_IN_AGENT else [SystemMessage(content=SYSTEM)]
    messages.append(HumanMessage(content=question))

    started = time.perf_counter()
    result = await agent.ainvoke({"messages": messages},
                                 config={"recursion_limit": MAX_AGENT_STEPS})
    elapsed = time.perf_counter() - started

    for message in result["messages"]:
        for call in getattr(message, "tool_calls", None) or []:
            args = call.get("args", {})
            print(f"   -> tool: {call['name']}({args})")
            sql = show_sql(call["name"], args)
            if not sql:
                continue
            print(f"      SQL: {sql}")
            ok, reason = validate(sql)
            verdict = "PASS" if ok else (f"FAIL: {reason}" if ok is False else f"n/a ({reason})")
            print(f"      validator: {verdict}")

    answer = message_text(result["messages"][-1].content)
    print(f"Q: {question}\nA: {answer}   [{elapsed:.1f}s]\n")
    return answer


for _question in [
    "What tables are available?",
    "How many bids are there in total?",
    "How many bids have the status Complete?",
    "How many suppliers do we have?",
]:
    await ask(_question)

### Reading the verdicts

- **`total_bids` / `bids_by_status` / `total_suppliers`** → **PASS** (they hit real business tables).
- **`list_tables` / `describe_table`** → **FAIL: references information_schema** — expected and harmless; those are safe system catalogs, not business tables, so the allowlist flags them.
- In **`prebuilt`** mode, the SQL printed is what **Gemini wrote**, and the verdict tells you whether it would have passed the guardrails — this is where validation actually matters.

**Reminder:** the verdict is informational only. Toolbox already ran the query; it did not consult this validator. To *enforce* it, you'd keep the `sql_expert` pipeline in front of the DB instead of exposing free-form `execute_sql`.

## Step 7 — Cleanup

In [ ]:
import shutil

# Copy LOG_PATH somewhere first if you still need the server log — RUN_DIR goes away.
print("server log tail:\n" + log_tail(10))

stop_toolbox()
shutil.rmtree(RUN_DIR, ignore_errors=True)   # removes tools.yaml and the log
print(f"\nToolbox stopped; {RUN_DIR} removed.")

## Taking this to production

Ordered by how much risk each step removes.

**1. A least-privilege database role is the only guardrail that cannot be talked around.**
Prompt rules are suggestions and `validate_sql()` is advisory — grants are enforced by Postgres itself.

```sql
CREATE ROLE rfp_readonly LOGIN PASSWORD :'pw';
GRANT CONNECT ON DATABASE sysco_rfp TO rfp_readonly;
GRANT USAGE ON SCHEMA public TO rfp_readonly;
GRANT SELECT ON bids, bids_old, suppliers, suppliers_old TO rfp_readonly;  -- allowlist, not ALL
ALTER ROLE rfp_readonly SET default_transaction_read_only = on;            -- no writes, ever
ALTER ROLE rfp_readonly SET statement_timeout = '15s';                     -- no runaway scans
ALTER ROLE rfp_readonly SET idle_in_transaction_session_timeout = '30s';
```

With that role in place, even `prebuilt` mode's free-form `execute_sql` cannot drop a table or lock the database. Point the Toolbox source at `rfp_readonly`, never at the application's read-write user.

**2. Expose views, not tables.** A `v_bid_summary` view instead of raw `bids` means PII and internal columns are unreachable by construction, and it gives you a place to put a `LIMIT`.

**3. Curated tools beat `execute_sql` on cost as well as safety** — fewer tokens, no schema-exploration round trips, deterministic query plans you can index for. When adding tools: one per business question; a `description` that says *when to call it* and *what it returns* (the model routes almost entirely on that text); typed parameters with realistic examples; no two tools that could both plausibly answer the same question; and keep a toolset under roughly 20 tools before routing accuracy starts to slip.

**4. Auth, if it ever leaves localhost.** `--address 127.0.0.1` is right for a notebook. A Toolbox reachable over a network without authenticated invocation (Cloud Run + IAM, or an auth service in `tools.yaml`) is an open SQL proxy.

**5. Observability.** Log every tool call as `{trace_id, tool, args, row_count, latency_ms, tokens}` through `logging_config.py`. Right now that information exists only as cell output and disappears when the kernel restarts.

**6. Evaluation.** Keep a fixed set of question → expected-answer pairs and assert on it in CI. `temperature=0` does not make an agent deterministic — a regression suite is how you find out that a model version or a prompt tweak broke tool routing.

**7. Failure modes to handle before users see them:** Toolbox process death (health check + restart), Vertex 429/503 (covered here by `max_retries`), the agent hitting `recursion_limit`, and a tool returning 100k rows — cap that in SQL, not in Python.

**8. Notebook hygiene.** Add `nbstripout` to `.pre-commit-config.yaml`. Notebook outputs are committed by default, and they are the easiest way to leak a hostname, a row of customer data, or a token into git.

> The same pipeline with the logic extracted into an importable, unit-tested module lives in `toolbox_runtime.py` + `mcp_gcp_toolbox_query_expert_v4.ipynb`. Use that shape when this graduates from a notebook into the FastAPI app; keep this one as the self-contained explainer.